# Build Results Fact

1. Read silver `results` table
1. Read silver `sprints` table
1. Add new column `session_type` with values `RACE` or `SPRINT`
1. UNION `results` and `sprints`
1. Derive additional columns
    - is_win -> Indicates that the driver own the race
    - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
    - has_points -> Indicates that the driver has scored points
1. Write the transformed data to gold `fact_session_results` table



#### Entity Relationship Diagram - Formula1 Silver Schema

![Formula1 Silver Data.png](../../z-course-images/formula1-silver-data-erd.png "Formula1 Silver Data.png")


#### Entity Relationship Diagram - Formula1 Gold Schema

![Formula1 Gold Data.png](../../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read source tables
- `silver.results`
- `silver.sprints`

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .withColumn("session_type", F.lit("RACE"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
         .withColumn("session_type", F.lit("SPRINT"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

#### Step 2 - UNION `results` and `sprints`

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

#### Step 3 - Add dervied columns
1. is_win -> Indicates that the driver own the race
1. is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
1. has_points -> Indicates that the driver has scored points


In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.filter("season = 2025"))

season,round,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
2025,7,alpine,gasly,10,63,10,0.0,13,13,Finished,RACE,false,false,false
2025,1,mclaren,piastri,2,57,81,2.0,9,9,Finished,RACE,false,false,true
2025,23,aston_martin,stroll,18,55,18,0.0,17,17,Retired,RACE,false,false,false
2025,4,mercedes,antonelli,5,57,12,0.0,11,11,Finished,RACE,false,false,false
2025,10,mercedes,antonelli,4,70,12,15.0,3,3,Finished,RACE,false,true,true
2025,17,ferrari,leclerc,10,51,16,2.0,9,9,Finished,RACE,false,false,true
2025,7,mclaren,norris,4,63,4,18.0,2,2,Finished,RACE,false,true,true
2025,13,aston_martin,alonso,20,44,14,0.0,17,17,Finished,RACE,false,false,false
2025,15,rb,hadjar,4,72,6,15.0,3,3,Finished,RACE,false,true,true
2025,22,alpine,colapinto,15,49,43,0.0,15,15,Lapped,RACE,false,false,false


#### Step 4 - Write the transformed data to the `gold` `fact_session_results` table

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

season,round,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
1950,3,wetteroth,rathmann,28,122,76,0.0,24,24,+16 Laps,RACE,false,false,false
1950,4,lago,martin,9,19,8,0.0,14,R,Accident,RACE,false,false,false
1950,7,milano,bonetto,23,0,52,0.0,27,W,Withdrew,RACE,false,false,false
1951,1,lago,etancelin,12,39,4,0.0,10,10,+3 Laps,RACE,false,false,false
1951,4,alfa,fagioli,7,77,8,4.0,1,1,Finished,RACE,true,true,true
1951,4,ferrari,whitehead,20,1,24,0.0,23,R,Engine,RACE,false,false,false
1951,6,lago,rosier,15,19,84,0.0,8,8,+1 Lap,RACE,false,false,false
1951,8,maserati,godia,17,60,44,0.0,10,10,+10 Laps,RACE,false,false,false
1952,6,maserati,bonetto,10,1,105,0.0,25,D,Disqualified,RACE,false,false,false
1952,8,maserati,cantoni,23,75,50,0.0,11,11,+5 Laps,RACE,false,false,false
